# E1: Model and Knowledge

Varies **one factor**: the phrasing model and knowledge access (PRD §8.1). Everything else stays at the initial **Reference Configuration**: advisor role prompt, compliance `A+B+C`, data exposure `aggregated`.

| Arm | Phrasing model | Retrieval |
|---|---|---|
| e1-llama3 | Llama 3.1 8B Instruct | off |
| e1-llama3-rag | Llama 3.1 8B Instruct | on |
| e1-finchat | AdaptLLM finance-chat 7B (Llama-2-based) | off |
| e1-llama2 | Llama 2 7B Chat (control) | off |

The contrasts isolate:

- **Domain adaptation:** finance-chat vs. Llama 2.
- **Model generation:** Llama 2 vs. Llama 3.
- **Knowledge access:** ± RAG.

The NLU and Judge roles are fixed to Llama 3.1 8B in every arm, so classification does not change with the model under test. All chat models run with Q4_K_M quantisation and `num_ctx = 4096`, because Llama 2 and finance-chat have a 4k context.

**Before running:** put the Knowledge Base PDFs in `rag_corpus/pdfs/` and fill in `rag_corpus/metadata.csv`.

## 1. Setup

**Runtime:** *Runtime → Change runtime type → T4 GPU*. Use the same GPU type for **all** comparative runs. A different GPU or runtime changes seeded outputs.

**Data:** upload the whole `experiments` folder of the repository to Google Drive as `MyDrive/FinanceTool/experiments`. Adjust `DRIVE_DIR` if you use a different path. Results are written to `experiments/results/` on Drive, so a disconnect loses at most one item; re-running a cell resumes where it stopped.

In [ ]:
# --- Parameters -------------------------------------------------------------------------
DRIVE_DIR = "/content/drive/MyDrive/FinanceTool/experiments"
EXPECTED_GPU = "T4"          # comparative runs must use the same GPU type (None = no check)
SMOKE_TEST = False           # True: run only a few items to test the setup
SMOKE_ITEMS = [1, 3, 31, 40, 41, 56, 65]
SCORE_QUALITY = True         # SM-9 answer-quality scoring with the scorer model (slower)
SCORER_MODEL = "qwen2.5:7b-instruct-q4_K_M"   # different model family than any model under test (OQ1)

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import sys
from pathlib import Path
BASE = Path(DRIVE_DIR)
RESULTS = BASE / "results"
RESULTS.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(BASE))
assert (BASE / "finlab").exists(), f"finlab package not found under {BASE}"

!pip -q install pypdf==6.21.1 huggingface_hub pyyaml

from finlab import colab
colab.check_gpu(EXPECTED_GPU)
colab.install_ollama()
colab.start_server()

## 2. Configuration and arms

In [ ]:
from finlab.pipeline import ExperimentConfig
from finlab import runner

EXPERIMENT = "E1"
MODELS = {"llama3.1:8b-instruct-q4_K_M", "llama2:7b-chat-q4_K_M", colab.FINCHAT_TAG, "bge-m3", SCORER_MODEL}
NEEDS_RAG = True

REFERENCE = ExperimentConfig(id="reference-initial", phrasing_model="llama3.1:8b-instruct-q4_K_M", retrieval=True,
                             role_prompt="advisor", compliance="A+B+C", data_exposure="aggregated")

def arm(id, **kw):
    d = REFERENCE.as_dict(); d.update(id=id, **kw)
    return ExperimentConfig.from_dict(d)

ARMS = {
    "e1-llama3":     arm("e1-llama3", phrasing_model="llama3.1:8b-instruct-q4_K_M", retrieval=False),
    "e1-llama3-rag": arm("e1-llama3-rag", phrasing_model="llama3.1:8b-instruct-q4_K_M", retrieval=True),
    "e1-finchat":    arm("e1-finchat", phrasing_model=colab.FINCHAT_TAG, retrieval=False),
    "e1-llama2":     arm("e1-llama2", phrasing_model="llama2:7b-chat-q4_K_M", retrieval=False),
}

## 3. Models, Knowledge Base, test set

In [ ]:
import datetime as dt, json
import pandas as pd
from finlab.llm import Ollama, Decoding
from finlab.pipeline import ExperimentConfig, Lab
from finlab.rag import KnowledgeBase
from finlab import runner, metrics

client = Ollama()
for m in sorted(MODELS):
    colab.pull(m)
print(colab.write_or_check_lock(RESULTS / "models_lock.json", sorted(MODELS)))

kb = None
if NEEDS_RAG:
    kb = KnowledgeBase.build(BASE / "rag_corpus", client, "bge-m3", RESULTS / "cache")
    print("KB snapshot:", kb.snapshot_id, "| passages:", len(kb.passages), "| documents:", len(kb.titles))

lab = Lab(BASE, client, nlu_model="llama3.1:8b-instruct-q4_K_M", results_dir=RESULTS, kb=kb)
items = runner.load_items(BASE, include_redteam=True, ids=SMOKE_ITEMS if SMOKE_TEST else None)
print(len(items), "items;", "rule snapshot", lab.rules.snapshot_id)

## 4. Run all arms

In [ ]:
summary = {}
for arm_id, cfg in ARMS.items():
    print(f"\n=== {arm_id} ===")
    records = runner.run_arm(lab, cfg, items, RESULTS, EXPERIMENT)
    summary[arm_id] = runner.finish_arm(lab, records, RESULTS, EXPERIMENT, arm_id, SCORER_MODEL, SCORE_QUALITY)

winner, rows = metrics.select_winner(summary)
table = pd.DataFrame(rows).set_index("arm")
display(table)
print("Proposed winner:", winner)

## Decision

`select_winner` proposes a winner. Arms that exceed a counter-metric tolerance (SM-C1 ≤ 10 %, SM-C2 ≤ 15 %) are excluded. Among the rest, the order is: fewest guardrail violations (SM-3), then highest behavior accuracy (SM-4), then lowest draft mismatch rate (SM-1a), then highest answer quality (SM-9).

Check the table and the per-item view below. If you choose differently, set `CHOSEN` and give the reason in `CHOICE_NOTE`. The next experiment reads `results/<experiment>/winner.json`.

In [ ]:
CHOSEN = winner          # or e.g. "e1-llama3-rag"
CHOICE_NOTE = "automatic proposal accepted"
runner.write_winner(RESULTS, EXPERIMENT, CHOSEN, rows, ARMS)
wj = json.loads((RESULTS / EXPERIMENT / "winner.json").read_text())
wj["note"] = CHOICE_NOTE
(RESULTS / EXPERIMENT / "winner.json").write_text(json.dumps(wj, indent=2))
print("winner.json written:", CHOSEN)

## Per-item view (mismatches between outcome and expected behavior)

In [ ]:
rows_all = []
for arm_id in ARMS:
    for line in (RESULTS / EXPERIMENT / arm_id / "items_scored.jsonl").read_text().splitlines():
        r = json.loads(line)
        rows_all.append({"arm": arm_id, "item": r["item_id"], "category": r["category"], "expected": r["expected_behavior"],
                         "class": r["class"], "outcome": r["outcome"], "route": r["route"],
                         "quality": (r.get("quality") or {}).get("score")})
per_item = pd.DataFrame(rows_all)
mism = per_item[per_item["expected"] != per_item["outcome"]]
display(mism.pivot_table(index=["item", "category", "expected"], columns="arm", values="outcome", aggfunc="first"))